# DepthWizard on a free cloud GPU (Google Colab)

Runs the full DepthWizard app — upload page, processing and 3D viewer — on Colab's free
NVIDIA GPU, and prints a public **https link** you can open from any laptop or phone.

**How to use**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save.
2. `Runtime` → **Run all**.
3. Open the `https://….trycloudflare.com` link printed by the last cell.

Keep this tab open while you use the link: Colab stops the machine when the tab is idle
for a long time, and the link changes each time the notebook is restarted.
The 3D view itself always draws on the viewing computer's graphics; integrated graphics
get a lighter preset automatically.

In [ ]:
#@title 1 · Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
#@title 2 · Get DepthWizard and install what it needs (about 2-3 minutes)
%cd /content
!test -d depthwizard || git clone -q --depth 1 https://github.com/jawaharananth/depthwizard.git
%cd /content/depthwizard
# Colab already has a CUDA build of PyTorch; install everything else.
!pip -q install transformers safetensors huggingface-hub rasterio pyproj shapely duckdb \
     opencv-python-headless scipy matplotlib pillow onnxruntime 2>&1 | tail -n 2
# 3D model compression (optional: the app falls back to uncompressed models without it)
!npm install -g gltfpack >/dev/null 2>&1 && echo "gltfpack ok" || echo "gltfpack not installed (optional)"

In [ ]:
#@title 3 · Install the trained height model (run 03) and check it
import os, shutil, glob
os.chdir("/content/depthwizard")
src = "training/run03/ckpt/height_best"
dst = "checkpoints/height_best"
os.makedirs(dst, exist_ok=True)
for f in glob.glob(f"{src}/*"):
    shutil.copy(f, dst)
print(sorted(os.listdir(dst)))
import depth_model as dm
bb = dm.DepthBackbone(metric=True)
print("model device:", bb.device)

In [ ]:
#@title 4 · (Optional) prepared showcase scenes from your Google Drive
#@markdown Put `depthwizard_scenes.zip` (the folder `scenes/`) in your Drive's root to show the
#@markdown seven prepared places; skip this cell to use uploads only.
import os
use_drive = False #@param {type:"boolean"}
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    z = "/content/drive/MyDrive/depthwizard_scenes.zip"
    if os.path.exists(z):
        !unzip -q -o "$z" -d /content/depthwizard
        print("scenes:", sorted(os.listdir("/content/depthwizard/scenes")))
    else:
        print("depthwizard_scenes.zip not found in My Drive")

In [ ]:
#@title 5 · Start the app and get the public link (keep this cell running)
import subprocess, time, re, os, urllib.request
os.chdir("/content/depthwizard")
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)
server = subprocess.Popen(["python", "server.py", "--host", "0.0.0.0", "--port", "8080"],
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(240):                       # the model loads before the first answer
    try:
        urllib.request.urlopen("http://127.0.0.1:8080/", timeout=3); break
    except Exception:
        time.sleep(2)
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8080",
                           "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
print("\n" + "=" * 64)
print("  DepthWizard is live:", url)
print("  Upload page :", url + "/")
print("  3D viewer   :", url + "/viewer/index.html")
print("=" * 64 + "\n")
print("Keep this cell running. Server log: /content/server.log")
try:
    while server.poll() is None:
        time.sleep(30)
except KeyboardInterrupt:
    pass